# SEPO Stage 1 — SFT Warm Start (Colab)
**Model**: `google/gemma-3-4b-it`  
**Method**: QLoRA (4-bit base + LoRA rank 8)  
**Data**: `sepo_sft_data/` — 6400 IPD strategy demonstrations  
**Output**: PEFT adapter uploaded to `kartiinx/gemma-3-4b-sepo-sft-hf`

Runtime: **T4 GPU** — `Runtime > Change runtime type > T4`

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers accelerate peft bitsandbytes trl huggingface_hub datasets

In [ ]:
# ── Cell 3: HuggingFace login ─────────────────────────────────────────────────
from huggingface_hub import login
login()  # paste HF token with read+write access

In [ ]:
# ── Cell 4: Clone repo (grpo-stage2 branch) ───────────────────────────────────
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN_HERE"  # github.com/settings/tokens → classic → repo scope
!git clone -b grpo-stage2 https://{GITHUB_TOKEN}@github.com/kirankumarmanku/sepo.git
%cd sepo

In [ ]:
# ── Cell 6: Load model + tokenizer (LoRA, no quantization) ───────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType

MODEL_ID = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# float16 — fits in T4 15GB (~8GB model weights)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Gemma 3 is multimodal — requires token_type_ids during training.
# This collator injects all-zeros (pure text, no image tokens).
class TextOnlyCollator:
    def __init__(self, tokenizer):
        self.base = DataCollatorForLanguageModeling(tokenizer, mlm=False)
    def __call__(self, features):
        batch = self.base(features)
        batch["token_type_ids"] = torch.zeros_like(batch["input_ids"])
        return batch

collator = TextOnlyCollator(tokenizer)
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 7: Load SFT dataset ──────────────────────────────────────────────────
from datasets import Dataset
import json

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl("sepo_sft_data/train.jsonl")
valid_rows = load_jsonl("sepo_sft_data/valid.jsonl")

# Apply chat template to each example
def format_example(row):
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False
    )}

train_dataset = Dataset.from_list(train_rows).map(format_example)
valid_dataset = Dataset.from_list(valid_rows).map(format_example)

print(f"Train: {len(train_dataset)} examples")
print(f"Valid: {len(valid_dataset)} examples")
print("Sample:", train_dataset[0]["text"][:200])


In [ ]:
# ── Cell 8: SFT Training ──────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/sft_gemma3_ipd"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    bf16=False,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    max_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=collator,
)

print("Starting SFT training...")
trainer.train()

In [ ]:
# ── Cell 9: Check val loss curve — pick best checkpoint ───────────────────────
# Best checkpoint is auto-loaded (load_best_model_at_end=True)
# Look at the logged eval_loss values above.
# We want the checkpoint where val loss ~0.01 (not 0.000 — that's overfit)
print("Training complete. Best model loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 10: Save adapter + upload to HuggingFace ─────────────────────────────
# Creates a NEW HF repo: kartiinx/gemma-3-4b-sepo-sft-hf
# This is a clean PEFT adapter (HF-compatible, unlike the MLX-fused version)

ADAPTER_REPO = "kartiinx/gemma-3-4b-sepo-sft-hf"

# Save adapter locally
model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")

# Push to HF Hub (creates private repo automatically)
model.push_to_hub(ADAPTER_REPO, private=True)
tokenizer.push_to_hub(ADAPTER_REPO, private=True)

print(f"Adapter uploaded to: {ADAPTER_REPO}")
print("Use this repo as --model in grpo_sepo_colab.ipynb")

In [ ]:
# ── Cell 11: Quick sanity check — generate a sample action ────────────────────
from peft import PeftModel

model.eval()
test_messages = [
    {"role": "system", "content": "You are playing the Iterated Prisoner's Dilemma game.\n\nRules:\n- Each round you choose one of two actions: <SILENT> or <TESTIFY>\n- If both players choose <SILENT>: you each get 3 points\n- If you choose <TESTIFY> and opponent chooses <SILENT>: you get 5, opponent gets 0\n- If you choose <SILENT> and opponent chooses <TESTIFY>: you get 0, opponent gets 5\n- If both choose <TESTIFY>: you each get 1 point\n\nYour goal is to maximise your total score over all rounds.\nRespond with ONLY your action: <SILENT> or <TESTIFY>. Nothing else."},
    {"role": "user", "content": "Round 1 of 8.\nThis is the first round. No history yet.\n\nWhat is your action?"}
]

input_ids = tokenizer.apply_chat_template(
    test_messages, return_tensors="pt", add_generation_prompt=True
).to("cuda")

with torch.no_grad():
    out = model.generate(input_ids, max_new_tokens=8, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)

response = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)
print(f"Model response: '{response}'")
print("Expected: <SILENT> (cooperate — SEPO-optimal for round 1)")